<a href="https://colab.research.google.com/github/1hamzaiqbal/probing-llm-math/blob/performance-test/performance_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Math Problem Performance Test

This notebook tests the model on problems from problems.csv and analyzes performance by problem type and difficulty.

In [8]:
import os

repo_path = 'probing-llm-math'
if not os.path.exists(repo_path):
    !git clone --branch performance-test https://github.com/1hamzaiqbal/probing-llm-math.git
else:
    print(f"Repository '{repo_path}' already exists.")

Repository 'probing-llm-math' already exists.


## Setup and Model Loading

In [9]:
!pip install -q transformers accelerate pandas seaborn matplotlib tqdm

In [10]:
import torch
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load model and tokenizer
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
print(f"Loading model: {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    dtype=torch.float16
)

print("✅ Model loaded successfully!")

Loading model: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Model loaded successfully!


## Load Problems Dataset

In [11]:
# Load problems from CSV
df = pd.read_csv('/content/probing-llm-math/data/problems.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nTopics: {df['topic'].unique()}")
print(f"\nDifficulty levels: {df['difficulty'].unique()}")
print(f"\nProblems per topic-difficulty combination:")
print(df.groupby(['topic', 'difficulty']).size())

# Display first few problems
print("\nFirst few problems:")
df.head()

Dataset shape: (75, 5)

Columns: ['id', 'topic', 'difficulty', 'question', 'answer']

Topics: ['Algebra' 'Number Theory' 'Counting & Probability']

Difficulty levels: ['Level 1' 'Level 2' 'Level 3' 'Level 4' 'Level 5']

Problems per topic-difficulty combination:
topic                   difficulty
Algebra                 Level 1       5
                        Level 2       5
                        Level 3       5
                        Level 4       5
                        Level 5       5
Counting & Probability  Level 1       5
                        Level 2       5
                        Level 3       5
                        Level 4       5
                        Level 5       5
Number Theory           Level 1       5
                        Level 2       5
                        Level 3       5
                        Level 4       5
                        Level 5       5
dtype: int64

First few problems:


,id,topic,difficulty,question,answer
0,1,Algebra,Level 1,"If $x = 2$ and $y = 5$, then what is the value...",We have \[\frac{x^4 + 2y^2}{6} = \frac{2^4 + ...
1,2,Algebra,Level 1,The perimeter of a rectangular garden is 60 fe...,"If the length is $l$ and the width is $w$, the..."
2,3,Algebra,Level 1,The function $f(x)$ is defined by $f(x)=x^{2}-...,$f(4)=4^2-4=16-4=\boxed{12}$.
3,4,Algebra,Level 1,The sum of two numbers is $45$. Their differen...,"Let $x,y$ be the larger and smaller numbers, r..."
4,5,Algebra,Level 1,What is the value of $(2x + 5)^2$ when $x = 3$?,We have $(2x+5)^2 = (2\cdot 3 + 5)^2 = 11^2 = ...


## Helper Functions

In [12]:
def extract_boxed_answer(text):
    """Extract answer from \\boxed{} format in solution text"""
    if text is None:
        return None
    match = re.search(r'\\boxed\{([^}]+)\}', text)
    return match.group(1) if match else None

def generate_answer(question, max_new_tokens=512, return_hidden_states=False):
    """Generate model answer for a given question, optionally returning hidden states"""
    prompt = f"Solve the following math problem step by step:\n\n{question}\n\nSolution:"

    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = inputs.to(model.device)

    with torch.no_grad():
        if return_hidden_states:
            # Get outputs with hidden states
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                output_hidden_states=True,
                return_dict_in_generate=True
            )
            generated_text = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)
            answer = generated_text[len(tokenizer.decode(inputs['input_ids'][0], skip_special_tokens=True)):].strip()

            # Extract hidden states from the last generated token
            # Note: hidden_states during generation is complex, so we'll do a separate forward pass
            with torch.no_grad():
                prompt_outputs = model(**inputs, output_hidden_states=True, use_cache=False)
                hidden_states = prompt_outputs.hidden_states
                layer_hidden = [h.squeeze(0).detach().cpu().numpy() for h in hidden_states]
                layer_means = [h.mean(axis=0) for h in layer_hidden]

            return answer, layer_means
        else:
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
            generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
            answer = generated_text[len(tokenizer.decode(inputs['input_ids'][0], skip_special_tokens=True)):].strip()
            return answer

def check_answer_match(model_answer, correct_answer_text):
    """Check if model answer matches the correct answer"""
    if correct_answer_text is None:
        return False

    correct = extract_boxed_answer(correct_answer_text)
    if correct is None:
        return False

    # Try to extract boxed answer from model's response
    model_boxed = extract_boxed_answer(model_answer)

    if model_boxed is not None:
        return model_boxed.strip() == correct.strip()

    # Check if correct answer appears in model's response
    return correct.strip() in model_answer

print("✅ Helper functions defined")

✅ Helper functions defined


## Run Performance Test

Evaluate the model on a subset of problems. Start with a small sample to test, then increase.

In [21]:
import pandas as pd
from tqdm.auto import tqdm

SEED = 42
SAMPLE_SIZE = 15  # change as needed

# sanity checks
required = {'question', 'answer', 'topic', 'difficulty'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns in df: {sorted(missing)}")

# group once
gb = df.groupby(['topic', 'difficulty'], dropna=False, group_keys=False)

# compute how many samples per group (at least 1, cap by group size)
num_groups = gb.size().shape[0]
per_group = max(1, SAMPLE_SIZE // max(1, num_groups))

eval_df = gb.apply(lambda g: g.sample(n=min(len(g), per_group), random_state=SEED), include_groups=False).reset_index(drop=True)

print(f"Evaluating on {len(eval_df)} problems...")

# only print distribution if columns present
if {'topic', 'difficulty'}.issubset(eval_df.columns):
    dist = eval_df.value_counts(['topic', 'difficulty']).sort_index()
    print("Distribution:\n", dist)
else:
    print("Skipping distribution print; stratifying columns not present in eval_df.")

predictions, is_correct = [], []
for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Evaluating"):
    pred = generate_answer(row['question'])
    correct = check_answer_match(pred, row['answer'])
    predictions.append(pred)
    is_correct.append(correct)

eval_df = eval_df.assign(model_answer=predictions, is_correct=is_correct)

overall_acc = float(eval_df['is_correct'].mean())
print(f"\n✅ Overall Accuracy: {overall_acc:.2%} ({eval_df['is_correct'].sum()}/{len(eval_df)})")


Evaluating on 15 problems...
Distribution:
 topic                   difficulty
Algebra                 Level 1       1
                        Level 2       1
                        Level 3       1
                        Level 4       1
                        Level 5       1
Counting & Probability  Level 1       1
                        Level 2       1
                        Level 3       1
                        Level 4       1
                        Level 5       1
Number Theory           Level 1       1
                        Level 2       1
                        Level 3       1
                        Level 4       1
                        Level 5       1
Name: count, dtype: int64


/tmp/ipython-input-4063017443.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  eval_df = gb.apply(lambda g: g.sample(n=min(len(g), per_group), random_state=SEED)).reset_index(drop=True)


Evaluating:   0%|          | 0/15 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Analysis by Topic and Difficulty

In [ ]:
# Accuracy by topic
print("📊 Accuracy by Topic:")
topic_acc = eval_df.groupby('topic')['is_correct'].agg(['mean', 'sum', 'count'])
topic_acc.columns = ['accuracy', 'correct', 'total']
topic_acc['accuracy'] = topic_acc['accuracy'].apply(lambda x: f"{x:.2%}")
print(topic_acc)

# Accuracy by difficulty
print("\n📊 Accuracy by Difficulty:")
diff_acc = eval_df.groupby('difficulty')['is_correct'].agg(['mean', 'sum', 'count'])
diff_acc.columns = ['accuracy', 'correct', 'total']
diff_acc['accuracy'] = diff_acc['accuracy'].apply(lambda x: f"{x:.2%}")
print(diff_acc)

## Heatmap: Performance by Topic and Difficulty

In [ ]:
# Create pivot table for heatmap
heatmap_data = eval_df.groupby(['topic', 'difficulty'])['is_correct'].mean().unstack(fill_value=np.nan)

# Reorder columns by difficulty
difficulty_order = ['Level 1', 'Level 2', 'Level 3', 'Level 4', 'Level 5']
heatmap_data = heatmap_data.reindex(columns=[col for col in difficulty_order if col in heatmap_data.columns])

# Create heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt='.0%',
    cmap='RdYlGn',
    vmin=0,
    vmax=1,
    cbar_kws={'label': 'Accuracy'},
    linewidths=0.5,
    linecolor='gray',
    mask=heatmap_data.isna()
)

plt.title('Model Accuracy by Problem Type and Difficulty', fontsize=14, fontweight='bold')
plt.xlabel('Difficulty Level', fontsize=12)
plt.ylabel('Problem Topic', fontsize=12)
plt.tight_layout()
plt.show()

print("\n📊 Heatmap Data (Accuracy Matrix):")
print(heatmap_data)

## Additional Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: Accuracy by Topic
topic_acc_values = eval_df.groupby('topic')['is_correct'].mean().sort_values(ascending=False)
axes[0].bar(range(len(topic_acc_values)), topic_acc_values.values, color='steelblue', alpha=0.7)
axes[0].set_xticks(range(len(topic_acc_values)))
axes[0].set_xticklabels(topic_acc_values.index, rotation=45, ha='right')
axes[0].set_ylabel('Accuracy', fontsize=11)
axes[0].set_title('Accuracy by Topic', fontsize=12, fontweight='bold')
axes[0].set_ylim([0, 1])
axes[0].grid(axis='y', alpha=0.3)

for i, v in enumerate(topic_acc_values.values):
    axes[0].text(i, v + 0.02, f'{v:.1%}', ha='center', va='bottom', fontweight='bold')

# Bar chart: Accuracy by Difficulty
diff_acc_values = eval_df.groupby('difficulty')['is_correct'].mean()
level_order = ['Level 1', 'Level 2', 'Level 3', 'Level 4', 'Level 5']
diff_acc_values = diff_acc_values.reindex([l for l in level_order if l in diff_acc_values.index])

axes[1].bar(range(len(diff_acc_values)), diff_acc_values.values, color='coral', alpha=0.7)
axes[1].set_xticks(range(len(diff_acc_values)))
axes[1].set_xticklabels(diff_acc_values.index, rotation=45, ha='right')
axes[1].set_ylabel('Accuracy', fontsize=11)
axes[1].set_title('Accuracy by Difficulty Level', fontsize=12, fontweight='bold')
axes[1].set_ylim([0, 1])
axes[1].grid(axis='y', alpha=0.3)

for i, v in enumerate(diff_acc_values.values):
    axes[1].text(i, v + 0.02, f'{v:.1%}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## Capture Hidden States (Sample)

Capture hidden states for a sample of problems to analyze later. This can help understand what the model is learning.

In [ ]:
# Sample problems for hidden state capture
# Try to get a diverse sample: some correct, some incorrect, across topics and difficulties
HIDDEN_STATE_SAMPLE_SIZE = 15

# Get stratified sample
correct_samples = eval_df[eval_df['is_correct'] == True].sample(min(len(eval_df[eval_df['is_correct'] == True]), HIDDEN_STATE_SAMPLE_SIZE // 2), random_state=42)
incorrect_samples = eval_df[eval_df['is_correct'] == False].sample(min(len(eval_df[eval_df['is_correct'] == False]), HIDDEN_STATE_SAMPLE_SIZE // 2), random_state=42)
hidden_state_samples = pd.concat([correct_samples, incorrect_samples]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Capturing hidden states for {len(hidden_state_samples)} problems...")
print(f"Distribution: {hidden_state_samples['is_correct'].value_counts().to_dict()}")

hidden_states_list = []
hidden_state_metadata = []

for idx, row in tqdm(hidden_state_samples.iterrows(), total=len(hidden_state_samples), desc="Capturing hidden states"):
    # Get hidden states by doing a forward pass on the question
    inputs = tokenizer(row['question'], return_tensors="pt")
    inputs = inputs.to(model.device)

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True, use_cache=False)
        hidden_states = outputs.hidden_states
        layer_hidden = [h.squeeze(0).detach().cpu().numpy() for h in hidden_states]
        layer_means = [h.mean(axis=0) for h in layer_hidden]

    hidden_states_list.append(layer_means)
    hidden_state_metadata.append({
        'id': row['id'],
        'topic': row['topic'],
        'difficulty': row['difficulty'],
        'is_correct': row['is_correct'],
        'question': row['question']
    })

# Convert to numpy array
hidden_states_array = np.array(hidden_states_list, dtype=np.float32)
print(f"\nHidden states shape: {hidden_states_array.shape}")
print(f"Format: (num_samples={hidden_states_array.shape[0]}, num_layers={hidden_states_array.shape[1]}, hidden_dim={hidden_states_array.shape[2]})")

# Save hidden states and metadata
np.savez_compressed(
    "performance_test_hidden_states.npz",
    hidden_states=hidden_states_array,
    metadata=np.array(hidden_state_metadata)
)

print("\n✅ Saved hidden states to 'performance_test_hidden_states.npz'")
print("   You can load this later for analysis with: np.load('performance_test_hidden_states.npz', allow_pickle=True)")

## Save Results

In [ ]:
# Save evaluation results
eval_df.to_csv('performance_test_results.csv', index=False)
print("✅ Saved evaluation results to 'performance_test_results.csv'")

# Save accuracy matrix
heatmap_data.to_csv('performance_test_accuracy_matrix.csv')
print("✅ Saved accuracy matrix to 'performance_test_accuracy_matrix.csv'")

# Summary statistics
print("\n" + "="*60)
print("EVALUATION SUMMARY")
print("="*60)
print(f"Total problems evaluated: {len(eval_df)}")
print(f"Overall accuracy: {eval_df['is_correct'].mean():.2%}")

topic_summary = eval_df.groupby('topic')['is_correct'].mean()
if len(topic_summary) > 0:
    print(f"\nBest performing topic: {topic_summary.idxmax()} ({topic_summary.max():.2%})")
    print(f"Worst performing topic: {topic_summary.idxmin()} ({topic_summary.min():.2%})")

diff_summary = eval_df.groupby('difficulty')['is_correct'].mean()
if len(diff_summary) > 0:
    print(f"\nBest performing difficulty: {diff_summary.idxmax()} ({diff_summary.max():.2%})")
    print(f"Worst performing difficulty: {diff_summary.idxmin()} ({diff_summary.min():.2%})")
print("="*60)

## Example: View Some Results

In [ ]:
# Show a few examples - some correct and some incorrect
print("Example correct answers:")
print("="*80)
for idx, row in eval_df[eval_df['is_correct'] == True].head(2).iterrows():
    print(f"\nProblem {row['id']} ({row['topic']}, {row['difficulty']})")
    print(f"Question: {row['question'][:150]}...")
    print(f"Expected: {extract_boxed_answer(row['answer'])}")
    print(f"Model: {row['model_answer'][:200]}...")
    print("-"*80)

print("\n\nExample incorrect answers:")
print("="*80)
for idx, row in eval_df[eval_df['is_correct'] == False].head(2).iterrows():
    print(f"\nProblem {row['id']} ({row['topic']}, {row['difficulty']})")
    print(f"Question: {row['question'][:150]}...")
    print(f"Expected: {extract_boxed_answer(row['answer'])}")
    print(f"Model: {row['model_answer'][:200]}...")
    print("-"*80)